In [ ]:
# ============================================================
# H1N1 VACCINE PREDICTION + USA HOSPITALISATION DATA ANALYSIS
# MSc Data Science Hackathon
# ============================================================

# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")


# ============================================================
# STEP 2: LOAD ORIGINAL DATASETS
# ============================================================

# Upload the three original CSV files from your computer.
from google.colab import files

uploaded = files.upload()

train = pd.read_csv("dataset_B_training.csv")
test = pd.read_csv("dataset_B_testing.csv")
sample_submission = pd.read_csv("dataset_B_example_submission.csv")

print("\nDATASETS LOADED SUCCESSFULLY")

print("Training data shape:", train.shape)
print("Testing data shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)


# ============================================================
# STEP 3: EXPLORE TRAINING DATA
# ============================================================

print("\nFIRST FIVE TRAINING RECORDS")
display(train.head())

print("\nCOLUMN NAMES")
print(train.columns.tolist())

print("\nDATA TYPES")
print(train.dtypes)

print("\nMISSING VALUES")
print(train.isnull().sum())

print("\nDUPLICATE ROWS")
print(train.duplicated().sum())

print("\nTARGET DISTRIBUTION")
print(train["h1n1_vaccine"].value_counts())

print("\nTARGET PERCENTAGES")
print(train["h1n1_vaccine"].value_counts(normalize=True) * 100)


# ============================================================
# STEP 4: PREPARE FEATURES AND TARGET
# ============================================================

X = train.drop(columns=["respondent_id", "h1n1_vaccine"])
y = train["h1n1_vaccine"]

X_test = test.drop(columns=["respondent_id"])

print("\nFEATURES:", X.shape)
print("TARGET:", y.shape)
print("TEST FEATURES:", X_test.shape)


# ============================================================
# STEP 5: IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNUMERICAL COLUMNS")
print(numerical_columns)

print("\nCATEGORICAL COLUMNS")
print(categorical_columns)


# ============================================================
# STEP 6: DATA PREPROCESSING
# ============================================================

# Numerical data:
# Fill missing values with median and scale features.

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical data:
# Fill missing values with most frequent value.
# Convert categories into numerical values.

try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

print("\nPREPROCESSING PIPELINE CREATED")


# ============================================================
# STEP 7: TRAIN AND VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTRAINING AND VALIDATION SPLIT")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


# ============================================================
# STEP 8: BUILD MACHINE LEARNING MODELS
# ============================================================

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

models = {
    "Logistic Regression": log_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

# XGBoost is optional. Install it if necessary.

try:
    from xgboost import XGBClassifier

    xgb_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        ))
    ])

    models["XGBoost"] = xgb_model

except ImportError:
    print("\nXGBoost not installed. Run: !pip install xgboost")


# ============================================================
# STEP 9: TRAIN MODELS AND EVALUATE PERFORMANCE
# ============================================================

results = []

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]

    accuracy = accuracy_score(y_val, predictions)
    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )
    f1 = f1_score(
        y_val, predictions, zero_division=0
    )
    auc = roc_auc_score(y_val, probabilities)

    cm = confusion_matrix(y_val, predictions)

    specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1])

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Specificity": specificity,
        "ROC AUC": auc
    })

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_val, predictions, zero_division=0
    ))

# Display comparison

comparison = pd.DataFrame(results)

print("\nMODEL PERFORMANCE COMPARISON")
display(comparison.round(4))

comparison.to_csv(
    "model_performance_comparison.csv",
    index=False
)


# ============================================================
# STEP 10: LOAD USA CDC HOSPITALISATION DATA
# ============================================================

print("\nLOADING USA CDC DATASET")

cdc_url = (
    "https://data.cdc.gov/resource/kvib-3txy.csv"
    "?$limit=50000"
)

try:
    hospital_df = pd.read_csv(cdc_url)

    print("\nCDC DATASET LOADED SUCCESSFULLY")

    print("Rows and columns:", hospital_df.shape)

    print("\nCOLUMN NAMES")
    print(hospital_df.columns.tolist())

    print("\nFIRST 10 RECORDS")
    display(hospital_df.head(10))

    print("\nDATA TYPES")
    print(hospital_df.dtypes)

    print("\nMISSING VALUES")
    print(hospital_df.isnull().sum())

    print("\nDUPLICATE ROWS")
    print(hospital_df.duplicated().sum())

except Exception as e:
    print("Could not load CDC dataset:", e)


# ============================================================
# STEP 11: CHECK INFLUENZA-RELATED RECORDS
# ============================================================

if "hospital_df" in globals():

    print("\nCHECKING CDC DATASET FOR INFLUENZA")

    possible_columns = [
        col for col in hospital_df.columns
        if any(word in col.lower() for word in [
            "virus", "pathogen", "condition",
            "respiratory", "influenza", "rate"
        ])
    ]

    print("Potential relevant columns:")
    print(possible_columns)

    # Display unique values for possible category columns.

    for col in possible_columns:
        if hospital_df[col].dtype == "object":
            print("\nValues in", col)
            print(hospital_df[col].dropna().unique()[:20])


# ============================================================
# STEP 12: SAVE ANALYSIS OUTPUTS
# ============================================================

print("\nSAVING ANALYSIS FILES")

comparison.to_csv(
    "model_performance_comparison.csv",
    index=False
)

if "hospital_df" in globals():
    hospital_df.to_csv(
        "cdc_hospitalisation_data.csv",
        index=False
    )

print("\nALL COMPLETED STEPS FINISHED!")
print("Check the generated CSV files in Colab.")

Libraries imported successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')